# Week 9 Phase 1.7 — physics ridge + 4D residual GP

Phase 1.5 showed that log(h) captures most global separation but misses some near-boundary structure. This notebook tests one focused remedy under the exact frozen protocol.

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT=Path.cwd().parents[1]
OUT=ROOT/'outputs/week9_phase1_7_physics_ridge_residual_gp'
TABLES=OUT/'tables'

## 1. Frozen baseline gate

The study stops unless all 100 split IDs and initial 16-point designs match Week 8.5. B1/q20/q30 remain evaluation-only.

In [2]:
json.loads((OUT/'baseline_gate.json').read_text())

{'frozen_protocol_sha256': 'bb16865a06d8fbdeea00f8c41f0929bfb7fbaf2f7b3e4ebade9cb2ffc59b1c66',
 'initial_design_exact_matches': 100,
 'margin_q20_AULC_16_80': 0.8135202205882354,
 'matched_random_q20_AULC_16_80': 0.7762204350490195,
 'outer_runs': 100,
 'split_exact_matches': 100,
 'starting_sha': 'b36a815e12dfa872c20a050228e0545f18c443ef',
 'status': 'PASS'}

## 2. Exact additive model

We fit `f=beta_0+beta_h z(log h)+r(P,VX,LS,ST)` with a logistic likelihood. The linear physics covariance and the 4D Matérn-3/2 covariance are added. Unlike a 5D GPC, the residual kernel never sees log(h), and there are no h–4D kernel interactions. Residual latent SD is restricted to [0.05,1.0].

![Model mechanism](../../outputs/week9_phase1_7_physics_ridge_residual_gp/figures/01_model_mechanism.png)

## 3. Static sanity check

This is diagnostic only: it asks whether the additive model captures some structure between h-only and the flexible 4D GPC.

In [3]:
s=pd.read_csv(TABLES/'static_model_summary.csv')
s[s.model.isin(['log_h_logistic','physics_ridge_residual_gp','gpc_4d'])][['model','subset','mean_roc_auc','mean_pr_auc','mean_balanced_accuracy','mean_keyhole_recall']]

,model,subset,mean_roc_auc,mean_pr_auc,mean_balanced_accuracy,mean_keyhole_recall
0,gpc_4d,B1_q20,0.919710,0.867253,0.815909,0.732327
1,gpc_4d,B1_q30,0.952277,0.897455,0.848762,0.761333
2,gpc_4d,full81,0.992587,0.968286,0.925087,0.866438
3,log_h_logistic,B1_q20,0.903899,0.804413,0.810431,0.709549
4,log_h_logistic,B1_q30,0.937929,0.834347,0.842876,0.750197
5,log_h_logistic,full81,0.989697,0.950055,0.922196,0.860959
6,physics_ridge_residual_gp,B1_q20,0.910594,0.811526,0.819493,0.717183
7,physics_ridge_residual_gp,B1_q30,0.943788,0.843658,0.852812,0.756482
8,physics_ridge_residual_gp,full81,0.990993,0.956724,0.925716,0.864384


## 4. Primary active-learning result

The new model uses ordinary Binary Margin from its own combined probability. Inference resamples 20 repeat blocks and retains five folds together.

In [4]:
pd.read_csv(TABLES/'primary_AULC_summary.csv')

,endpoint,new_mean,baseline_mean,delta,ci_lower,ci_upper,positive_repeat_blocks,repeat_blocks,decision,bootstrap_method
0,B1_q20_AULC_16_80,0.833539,0.813520,0.020018,0.014071,0.026264,19,20,PASS,20 repeat blocks resampled; all five folds ret...
1,B1_q30_AULC_16_80,0.877056,0.866116,0.010941,0.005966,0.015978,15,20,secondary,20 repeat blocks resampled; all five folds ret...


![q20 learning curves](../../outputs/week9_phase1_7_physics_ridge_residual_gp/figures/02_q20_learning_curves.png)

![Matched AULC contrast](../../outputs/week9_phase1_7_physics_ridge_residual_gp/figures/03_primary_delta_AULC.png)

## 5. Budget 40 and missed Keyholes

Accuracy, balanced accuracy, recall, FN and FP are kept separate. Lower FN is the direct missed-Keyhole diagnostic.

In [5]:
t=pd.read_csv(TABLES/'terminal_budget_summary.csv')
t[(t.budget==40)&(t.subset=='B1_q20')][['model','accuracy','balanced_accuracy','keyhole_recall','false_negative','false_positive']]

,model,accuracy,balanced_accuracy,keyhole_recall,false_negative,false_positive
0,frozen_4d_margin,0.817059,0.797113,0.711044,1.91,1.20
6,physics_ridge_residual_gp,0.834118,0.812943,0.728079,1.76,1.06


## 6. Does the residual really correct h failures?

A class change is defined on latent signs: `sign(g)` versus `sign(g+r)`. Corrections and newly introduced errors are both counted.

In [6]:
d=pd.read_csv(TABLES/'model_mechanism_diagnostics.csv.gz')
cols=['budget','corrected_count','worsened_count','keyholes_recovered_by_residual','new_keyhole_misses_induced','physics_rms_latent','residual_rms_latent','residual_sd']
d[(d.subset=='B1_q20')&d.budget.isin([40,80])].groupby('budget')[cols[1:]].mean()

,corrected_count,worsened_count,keyholes_recovered_by_residual,new_keyhole_misses_induced,physics_rms_latent,residual_rms_latent,residual_sd
budget,,,,,,,
40,0.21,0.20,0.13,0.04,2.423964,0.182356,0.753644
80,0.37,0.21,0.14,0.02,3.022291,0.309013,0.935861


### Representative budget-40 q20 cases

These are repeated held-out summaries, not independent experiments.

In [7]:
r=pd.read_csv(TABLES/'representative_residual_cases.csv')
cols=['P','VX','LS_um','ST','log_h','mean_physics_probability','mean_residual_correction','mean_combined_probability','truth','corrected_occurrences']
r[r.corrected_occurrences>0].head(5)[cols]

,P,VX,LS_um,ST,log_h,mean_physics_probability,mean_residual_correction,mean_combined_probability,truth,corrected_occurrences
0,140.539865,0.255659,45.000000,300.000000,20.640719,0.448534,0.137554,0.483294,1,6
1,211.816523,0.520711,48.078260,318.660442,20.596021,0.384452,0.153741,0.429483,1,5
2,336.384536,0.857000,51.482757,367.720955,20.706809,0.603984,-0.176339,0.554782,0,3
3,149.217973,0.206405,48.737953,365.118948,20.687945,0.555644,-0.209337,0.503874,0,2
4,140.119888,0.222892,45.000000,300.000000,20.706304,0.545389,0.131502,0.567871,1,2


## 7. Label-free 2D view

PCA uses only standardized P,VX,LS,ST. Color and outlines are overlaid after projection; PCA is not a physical boundary or feature-importance method.

![PCA residual diagnostic](../../outputs/week9_phase1_7_physics_ridge_residual_gp/figures/04_pca_residual_diagnostic.png)

## 8. Safe conclusion

The preregistered primary comparison passes on this frozen simulator benchmark. The claim is about the combined additive model and Binary-Margin policy, not acquisition alone. The residual recovers some missed Keyholes, but its effect is modest and its amplitude cap binds frequently. No transfer, causality, safety, or guaranteed query-saving claim follows.